In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import numpy as np
import bt as bt
import pandas as pd
import yfinance as yf
from io import StringIO

In [2]:
# Configurando o driver do navegador
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# URL para FIIs no site Fundamentus ou outro de sua escolha
url = 'https://www.fundamentus.com.br/fii_resultado.php'

# Acessando a página
driver.get(url)
local_tabela = '/html/body/div[1]/div[2]/table'  # Atualize o caminho XPath se necessário
elemento = driver.find_element("xpath", local_tabela)
html_tabela = elemento.get_attribute('outerHTML')
driver.quit()

In [3]:
# Processando a tabela
tabela = pd.read_html(StringIO(html_tabela), thousands=".")[0]
tabela = tabela.set_index("Papel")
tabela

,Segmento,Cotação,FFO Yield,Dividend Yield,P/VP,Valor de Mercado,Liquidez,Qtd de imóveis,Preço do m2,Aluguel por m2,Cap Rate,Vacância Média
Papel,,,,,,,,,,,,
AAZQ11,Títulos e Val. Mob.,"7,21","14,90%","16,26%","0,84",173309000,501778,0,"0,00","0,00","0,00%","0,00%"
ABCP11,Shoppings,"84,85","8,22%","8,28%","0,79",399566000,81163,1,"5.570,60","567,85","10,19%","2,90%"
AEFI11,Outros,"174,90","8,55%","0,00%","1,75",411893000,0,0,"0,00","0,00","0,00%","0,00%"
AFCR11,Híbrido,"103,15","12,61%","0,00%","1,07",498867000,0,0,"0,00","0,00","0,00%","0,00%"
AFHI11,Títulos e Val. Mob.,"91,46","11,75%","12,29%","0,97",416657000,821741,0,"0,00","0,00","0,00%","0,00%"
...,...,...,...,...,...,...,...,...,...,...,...,...
YUFI11,Residencial,"59,50","3,73%","3,40%","0,79",29312700,0,38,"520,68","223,02","42,83%","0,00%"
ZAGH11,Híbrido,"9,40","5,80%","8,05%","1,00",83581600,19872,1,"2.318,78","81,00","3,49%","0,00%"
ZAVC11,Títulos e Val. Mob.,"9,10","14,43%","16,23%","0,91",28185900,18825,0,"0,00","0,00","0,00%","0,00%"


In [4]:
# Função para limpar colunas (igual à sua abordagem)
def clean_column(col):
    if col.dtype == 'object':
        col = col.str.replace("%", "", regex=False)
        col = col.str.replace(".", "", regex=False)
        col = col.str.replace(",", ".", regex=False)
        col = pd.to_numeric(col, errors='coerce')
    return col

# Limpando as colunas
for col in tabela.columns:
    tabela[col] = clean_column(tabela[col])

# Filtrando FIIs com base em critérios
tabela = tabela[['Cotação', 'Dividend Yield', 'P/VP', 'Vacância Média', 'Liquidez']]
tabela = tabela[tabela['Liquidez'] > 1000000]  # Liquidez mínima
tabela = tabela[tabela['Dividend Yield'] > 8]  # Dividend Yield superior a 6%
tabela = tabela[tabela['P/VP'] > 0.8]  # Preço/Valor Patrimonial < 1.2
tabela = tabela[tabela['Vacância Média'] < 15]  # Vacância menor que 10%

# Rankeando os FIIs
tabela['ranking_dy'] = tabela['Dividend Yield'].rank(ascending=False)
tabela['ranking_pvp'] = tabela['P/VP'].rank(ascending=True)
tabela['ranking_vac'] = tabela['Vacância Média'].rank(ascending=True)
tabela['ranking_total'] = tabela['ranking_dy'] + tabela['ranking_pvp'] + tabela['ranking_vac']
tabela = tabela.sort_values('ranking_total')

# Exibindo os FIIs rankeados
print(tabela)

        Cotação  Dividend Yield  P/VP  Vacância Média  Liquidez  ranking_dy  \
Papel                                                                         
JSAF11     7.75           14.15  0.87            0.00   1610430         8.0   
IRDM11    67.64           13.65  0.82            0.00   2700520        15.0   
VCJR11    81.80           14.00  0.87            0.00   1661020         9.5   
FGAA11     8.61           18.34  0.91            0.00   1045730         1.0   
VGHF11     7.56           13.98  0.88            0.00   2849020        11.0   
RVBI11    64.11           13.49  0.85            0.00   1124400        17.0   
VGIP11    81.34           14.00  0.90            0.00   1446900         9.5   
CVBI11    83.10           13.78  0.90            0.00   2018520        12.0   
CPTS11     7.25           12.71  0.83            0.00   7800290        25.0   
RZAG11     8.84           15.22  0.92            0.00   1415990         5.0   
CPSH11     9.65           12.40  0.81            0.0

In [5]:
start = "2018-1-1"
end = "2024-11-1"

In [6]:
tabela = tabela.head(6)
tickers = tabela.index
tickers = [fii + '.SA' for fii in tickers]
wallet = yf.download(tickers, start, end)["Adj Close"]
wallet = wallet.dropna()

wallet

Failed to get ticker 'RVBI11.SA' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'JSAF11.SA' reason: Expecting value: line 1 column 1 (char 0)
[                       0%                       ]Failed to get ticker 'VGHF11.SA' reason: Expecting value: line 1 column 1 (char 0)
[****************      33%                       ]  2 of 6 completedFailed to get ticker 'IRDM11.SA' reason: Expecting value: line 1 column 1 (char 0)
[*********************100%***********************]  6 of 6 completed

6 Failed downloads:
['FGAA11.SA', 'VCJR11.SA']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
['RVBI11.SA', 'JSAF11.SA', 'VGHF11.SA', 'IRDM11.SA']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')


Ticker,FGAA11.SA,IRDM11.SA,JSAF11.SA,RVBI11.SA,VCJR11.SA,VGHF11.SA
Date,,,,,,


In [7]:
# Obter os preços históricos
data = bt.get(tickers, start='2018-01-01', end='2024-11-01')

Failed to get ticker 'JSAF11.SA' reason: Expecting value: line 1 column 1 (char 0)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['JSAF11.SA']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')
Failed to get ticker 'IRDM11.SA' reason: Expecting value: line 1 column 1 (char 0)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['IRDM11.SA']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['VCJR11.SA']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['FGAA11.SA']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
Failed to get ticker 'VGHF11.SA' reason: Expecting value: line 1 column 1 (char 0)
[*********************100%***********************]  1 of 1 completed

1 Faile

In [8]:
# Passo 1: Calcular os pesos otimizados
precos = data.copy()
retornos = precos.pct_change().apply(lambda x: np.log(1 + x)).dropna()
media_retornos = retornos.mean()
matriz_cov = retornos.cov()

# Configurações
numero_carteiras = 10000
tabela_retornos_esperados = np.zeros(numero_carteiras)
tabela_volatilidades_esperadas = np.zeros(numero_carteiras)
tabela_sharpe = np.zeros(numero_carteiras)
tabela_pesos = np.zeros((numero_carteiras, len(precos.columns)))

# Simulação de carteiras
for k in range(numero_carteiras):
    pesos = np.random.random(len(precos.columns))
    pesos = pesos / np.sum(pesos)
    tabela_pesos[k, :] = pesos
    
    tabela_retornos_esperados[k] = np.sum(media_retornos * pesos * 252)
    tabela_volatilidades_esperadas[k] = np.sqrt(np.dot(pesos.T, np.dot(matriz_cov * 252, pesos)))
    tabela_sharpe[k] = tabela_retornos_esperados[k] / tabela_volatilidades_esperadas[k]

# Carteira com Sharpe máximo
indice_do_sharpe_maximo = tabela_sharpe.argmax()
pesos_otimizados = tabela_pesos[indice_do_sharpe_maximo]

pesos_otimizados

C:\Users\Emanuel Venditto\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\numpy\lib\_function_base_impl.py:562: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
C:\Users\Emanuel Venditto\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\numpy\_core\_methods.py:139: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\Emanuel Venditto\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\core\frame.py:11211: RuntimeWarning: Degrees of freedom <= 0 for slice
  base_cov = np.cov(mat.T, ddof=ddof)
C:\Users\Emanuel Venditto\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\numpy\lib\_function_base_impl.py:2848: RuntimeWarning: divide by zer

array([0.1448023 , 0.21747132, 0.06058916, 0.38885751, 0.13097265,
       0.05730706])

In [9]:
# Converter para dicionário (formato necessário para WeighSpecified)
ativos = precos.columns
pesos_dicionario = dict(zip(ativos, pesos_otimizados))
print("Pesos otimizados:", pesos_dicionario)

Pesos otimizados: {'jsaf11sa': np.float64(0.1448022993456884), 'irdm11sa': np.float64(0.21747131555397928), 'vcjr11sa': np.float64(0.060589157289917754), 'fgaa11sa': np.float64(0.3888575139438129), 'vghf11sa': np.float64(0.1309726531718405), 'rvbi11sa': np.float64(0.05730706069476116)}


In [10]:
# Passo 2: Usar os pesos na estratégia BT
strategy1 = bt.Strategy('Optimized Portfolio',
                       [bt.algos.RunMonthly(),
                        bt.algos.SelectAll(),
                        bt.algos.WeighSpecified(**pesos_dicionario),
                        bt.algos.Rebalance()])
# Estratégia com pesos ajustados pelo risco (WeighRiskAdjusted)
# Aqui você pode definir a estratégia ajustada pelo risco com base em sua preferência.
# Por exemplo, uma forma de ajustar os pesos pode ser através da inversão da volatilidade.
# Isso significa que ativos mais voláteis terão pesos menores.

# Calcular a volatilidade dos ativos
volatilidades = np.sqrt(np.diag(matriz_cov))  # Volatilidade individual dos ativos (desvio padrão)
volatilidades_invertidas = 1 / volatilidades  # Inversão da volatilidade para o ajuste de risco

# Normalizar os pesos ajustados pelo risco
pesos_ajustados = volatilidades_invertidas / np.sum(volatilidades_invertidas)
pesos_ajustados_dicionario = dict(zip(tickers, pesos_ajustados))


# Estratégia com pesos ajustados pelo risco (WeighRiskAdjusted)
strategy2 = bt.Strategy('Risk Adjusted Portfolio',
                                     [bt.algos.RunMonthly(),
                                      bt.algos.SelectAll(),
                                      bt.algos.WeighSpecified(**pesos_ajustados_dicionario),
                                      bt.algos.Rebalance()])

In [11]:
# Passo 2: Usar os pesos na estratégia BT
strategy1 = bt.Strategy('Optimized Portfolio',
                       [bt.algos.RunMonthly(),
                        bt.algos.SelectAll(),
                        bt.algos.WeighSpecified(**pesos_dicionario),
                        bt.algos.Rebalance()])
# Estratégia com pesos ajustados pelo risco (WeighRiskAdjusted)
# Aqui você pode definir a estratégia ajustada pelo risco com base em sua preferência.
# Por exemplo, uma forma de ajustar os pesos pode ser através da inversão da volatilidade.
# Isso significa que ativos mais voláteis terão pesos menores.

# Calcular a volatilidade dos ativos
volatilidades = np.sqrt(np.diag(matriz_cov))  # Volatilidade individual dos ativos (desvio padrão)
volatilidades_invertidas = 1 / volatilidades  # Inversão da volatilidade para o ajuste de risco

# Normalizar os pesos ajustados pelo risco
pesos_ajustados = volatilidades_invertidas / np.sum(volatilidades_invertidas)
pesos_ajustados_dicionario = dict(zip(tickers, pesos_ajustados))


# Estratégia com pesos ajustados pelo risco (WeighRiskAdjusted)
strategy2 = bt.Strategy('Risk Adjusted Portfolio',
                                     [bt.algos.RunMonthly(),
                                      bt.algos.SelectAll(),
                                      bt.algos.WeighSpecified(**pesos_ajustados_dicionario),
                                      bt.algos.Rebalance()])

In [12]:
# Passo 3: Criar e rodar os backtests
backtest_optimized = bt.Backtest(strategy1, data)
backtest_risk_adjusted = bt.Backtest(strategy2, data)

# Rodar os backtests para as três estratégias
result = bt.run(backtest_optimized, backtest_risk_adjusted)

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
# Exibir os resultados comparativos
result.plot()
result.display()